# Lab 06-01 — Cross-encoder reranking: retrieve wide, re-score short

**Track 06 · Re-ranking** — the two-stage pattern: a cheap bi-encoder scans the whole index wide, then a precise cross-encoder re-scores only the short candidate list.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the wide retrieval, and the cross-encoder re-scoring all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The bi-encoder embedder (BGE) maps query and passage to single vectors and scores them with cosine — fast enough to scan the whole index, but blind to fine-grained relevance: a passage sharing the topic scores well even when it does not answer the question. A cross-encoder reads the (query, passage) PAIR as one input, so it sees every token of both at once — far better at separating "on topic" from "actually answers". The catch: one forward pass per pair, which is why the cross-encoder only ever sees a SHORT candidate list.

This lab wires the two together on nfcorpus (BEIR medical FAQ corpus):

* **stage 1** — bi-encoder (BGE) retrieves a wide top-20 candidate list;
* **stage 2** — a sentence-transformers `CrossEncoder` re-scores the 20 candidates and keeps the top 3.

For each question we track the position of the gold (qrels-relevant) passage before (bi top-20) and after (rerank top-3) to show the lift. Demo queries are picked deterministically: the first qrels queries whose gold passage the bi-encoder put OUTSIDE its top-3 but INSIDE its top-20 — the cases where a reranker earns its keep.


## Setup

One prerequisite must hold before this notebook will run:

- **nfcorpus on disk** — `Data/corpus/beir-nfcorpus/nfcorpus/` (BEIR medical FAQ corpus: `corpus.jsonl` + `queries.jsonl` + `qrels/test.tsv`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `sentence-transformers`, and `faiss-cpu`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from sentence_transformers import CrossEncoder  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `NFCORPUS_DIR` points at the nfcorpus subset already on disk; `NF_N_DOCS = 600` takes a deterministic head of the 3633-doc corpus (no randomness, reproducible runs); `NF_MAX_QUERIES = 400` bounds the search pool to the first 400 qrels-covered queries; `WIDE_K = 20` is the wide stage-1 candidate list, `TOP_K = 3` the stage-2 shortlist, and `N_DEMO = 3` how many lift-queries to collect. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
NFCORPUS_DIR = Path("Data/corpus/beir-nfcorpus/nfcorpus")
CORPUS_PATH = NFCORPUS_DIR / "corpus.jsonl"
QUERIES_PATH = NFCORPUS_DIR / "queries.jsonl"
QRELS_PATH = NFCORPUS_DIR / "qrels" / "test.tsv"
NF_N_DOCS = 600  # deterministic head of the 3633-doc nfcorpus corpus
NF_MAX_QUERIES = 400  # search pool: first 400 qrels-covered query ids
WIDE_K = 20  # stage-1 bi-encoder candidate list
TOP_K = 3  # stage-2 cross-encoder shortlist
N_DEMO = 3  # how many lift-queries to show (and gate)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
CE_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — nfcorpus corpus + queries + qrels

`load_nfcorpus` reads the first `n` corpus docs as `title + " " + text` — nfcorpus titles are short keyword phrases ("Breast Cancer Cells Feed on Cholesterol") that the queries are written against, so they belong in the indexed text. `load_nf_queries` returns `(query_id, query_text)` pairs in file order, and `load_qrels` keeps only the qrels rows with score >= 1 as `{query_id: {relevant_corpus_id, ...}}`. Two small helpers round out the section: `preview` flattens a passage onto one line for printing, and `gold_rank` returns the 1-based position of the first gold doc in a ranked list (or None).


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — nfcorpus corpus + queries + qrels
# --------------------------------------------------------------------------
def load_nfcorpus(path: Path, n: int) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for the first ``n`` corpus docs.

    Field choice: ``title + " " + text`` — nfcorpus titles are short keyword
    phrases ("Breast Cancer Cells Feed on Cholesterol") that the queries are
    written against, so they belong in the indexed text.
    """
    texts: list[str] = []
    ids: list[str] = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            doc = json.loads(line)
            ids.append(doc["_id"])
            texts.append(f'{doc["title"]} {doc["text"]}')
    return texts, ids


def load_nf_queries(path: Path) -> list[tuple[str, str]]:
    """Return [(query_id, query_text)] for every query, in file order."""
    out: list[tuple[str, str]] = []
    with open(path) as f:
        for line in f:
            q = json.loads(line)
            out.append((q["_id"], q["text"]))
    return out


def load_qrels(path: Path) -> dict[str, set[str]]:
    """Return {query_id: {relevant_corpus_id, ...}} (qrels score >= 1)."""
    qrels: dict[str, set[str]] = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3 or parts[0] == "query-id":
                continue  # header row
            qid, cid = parts[0], parts[1]
            if int(parts[2]) >= 1:
                qrels.setdefault(qid, set()).add(cid)
    return qrels


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def gold_rank(docs: list[Document], gold_ids: set[str]) -> int | None:
    """1-based position of the first gold doc in ``docs``, else None."""
    for i, doc in enumerate(docs, start=1):
        if doc.metadata.get("id") in gold_ids:
            return i
    return None


## 3. Experiment — embed, index, retrieve wide, rerank short

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 600 docs once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. Stage 1 is a plain `similarity_search_by_vector` at `WIDE_K = 20` (cheap, whole-index scan); stage 2 is a sentence-transformers `CrossEncoder` scored inline — one `predict` call over the 20 pairs, scores attached to `metadata["score"]`, sorted descending, top `TOP_K = 3` kept. This is the same mechanism the shared `src/tools/reranker.py` class wraps.

A query qualifies as a demo query when its gold passage sits in the bi top-20 but OUTSIDE the bi top-3, and the cross-encoder then promotes it back into its top-3 — the lift cases where the reranker earns its keep. The search pool is the first `NF_MAX_QUERIES` qrels-covered queries; we stop at `N_DEMO`.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, index, retrieve wide, rerank short
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


def _rerank(query: str, documents: list[Document], ce_model: CrossEncoder,
            top_k: int = TOP_K, batch_size: int = 32) -> list[Document]:
    """Score (query, doc) pairs, attach metadata["score"], keep top-k."""
    if not documents:
        return []
    pairs = [(query, doc.page_content) for doc in documents]
    scores = ce_model.predict(pairs, show_progress_bar=False, batch_size=batch_size)
    for doc, score in zip(documents, scores):
        doc.metadata["score"] = float(score)
    ranked = sorted(documents, key=lambda d: d.metadata["score"], reverse=True)
    return ranked[:top_k]


def run_experiment() -> dict:
    nf_texts, nf_ids = load_nfcorpus(CORPUS_PATH, NF_N_DOCS)
    nf_queries = load_nf_queries(QUERIES_PATH)
    qrels = load_qrels(QRELS_PATH)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    nf_vecs = embedder.embed_documents(nf_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(nf_texts, nf_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedding=_PrecomputedEmbeddings(nf_texts, nf_vecs))
    index_s = time.perf_counter() - t0

    # --- The two stages ------------------------------------------------------
    ce_model = CrossEncoder(CE_MODEL_NAME)  # stage 2: short + precise

    # Demo queries: gold in bi top-20 but NOT in bi top-3, and recovered into
    # the cross-encoder top-3 — the lift cases where the reranker earns its
    # keep. The cross-encoder only scores the rare candidates that pass the
    # cheap bi-encoder filter, so the search pool can be large.
    subset_ids = set(nf_ids)
    covered = [(qid, q) for qid, q in nf_queries if qid in qrels][:NF_MAX_QUERIES]
    t0 = time.perf_counter()
    results = []
    for qid, qtext in covered:
        gold = qrels[qid] & subset_ids
        if not gold:
            continue
        wide = store.similarity_search_by_vector(embedder.embed_query(qtext), k=WIDE_K)
        g_before = gold_rank(wide, gold)
        if g_before is None or g_before <= TOP_K:
            continue  # not a lift case: gold already in bi top-3 (or missing)
        reranked = _rerank(qtext, wide, ce_model, top_k=TOP_K)
        g_after = gold_rank(reranked, gold)
        if g_after is None or g_after > TOP_K:
            continue  # cross-encoder did not recover it — not a demo query
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "gold_ids": gold,
                "g_before": g_before,
                "g_after": g_after,
                "reranked": reranked,
                "scores": [d.metadata["score"] for d in reranked],
            }
        )
        if len(results) >= N_DEMO:
            break
    rerank_s = time.perf_counter() - t0

    return {
        "nf_ids": nf_ids,
        "demo": [(r["qid"], r["question"], r["gold_ids"]) for r in results],
        "results": results,
        "indexed": len(nf_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "rerank_s": rerank_s,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/index timings; per query, the gold-position lift — rank BEFORE in the bi top-20 vs rank AFTER in the rerank top-3, the descending rerank scores, and a preview of the new top-1; then a takeaway explaining why the cross-encoder catches the fine-grained relevance the pooled cosine vector smears away — and why that precision costs one forward pass per pair, so it must never see more than a short candidate list.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 01 — Cross-encoder reranking: retrieve wide, re-score short")
    print(f"{BGE_MODEL_NAME} (bi-encoder, top-{WIDE_K}) -> "
          f"{CE_MODEL_NAME} (top-{TOP_K})")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} nfcorpus docs (first {NF_N_DOCS} of 3633)")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")

    print(f"\n[2] Gold-position lift (bi-encoder top-{WIDE_K} -> cross-encoder top-{TOP_K}):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      gold docs: {sorted(r['gold_ids'])}")
        print(f"      gold rank BEFORE (bi top-{WIDE_K}): "
              f"{r['g_before'] if r['g_before'] is not None else 'missing'}")
        print(f"      gold rank AFTER  (rerank top-{TOP_K}): "
              f"{r['g_after'] if r['g_after'] is not None else 'missing'}")
        print(f"      rerank scores (desc): {[f'{s:.3f}' for s in r['scores']]}")
        top = r["reranked"][0]
        print(f"      top-1: [{top.metadata['id']}] {preview(top.page_content)}")

    print(f"\n[3] Takeaway")
    print("    The bi-encoder put every gold passage OUTSIDE its top-3")
    print("    (otherwise the question would not be in this demo). The")
    print("    cross-encoder reads each (query, passage) pair as one input,")
    print("    so it can catch the fine-grained relevance the pooled cosine")
    print("    vector smears away — and promotes the gold into the top-3.")
    print("    That precision costs one forward pass per pair, which is why")
    print("    the cross-encoder must never see more than a short candidate")
    print(f"    list (here {WIDE_K}); scanning the whole corpus pair-wise would be")
    print("    orders of magnitude slower than the bi-encoder.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `NF_N_DOCS` docs indexed, exactly `N_DEMO` lift-queries selected, and for every demo query — gold was outside the bi top-3 before, promoted into the rerank top-3 after, strictly improved, and rerank scores are descending. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"exactly {NF_N_DOCS} nfcorpus docs indexed", exp["indexed"] == NF_N_DOCS))
    checks.append((f"{N_DEMO} lift-queries selected (gold in bi top-{WIDE_K}, "
                   f"outside bi top-{TOP_K})", len(exp["results"]) == N_DEMO))

    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} gold was outside bi top-{TOP_K} before",
                       r["g_before"] is not None and r["g_before"] > TOP_K))
        checks.append((f"{tag} gold promoted into rerank top-{TOP_K}",
                       r["g_after"] is not None and r["g_after"] <= TOP_K))
        checks.append((f"{tag} gold rank strictly improved",
                       r["g_after"] is not None and r["g_before"] is not None
                       and r["g_after"] < r["g_before"]))
        checks.append((f"{tag} rerank scores are descending",
                       all(a >= b for a, b in zip(r["scores"], r["scores"][1:]))))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes of embedding + ~60 cross-encoder scoring passes on nfcorpus — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three demo queries whose gold passage the bi-encoder missed in its top-3 but caught in its top-20 — the cases where the reranker earns its keep. Each row shows gold position before (bi top-20) and after (rerank top-3).


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the nfcorpus files are intact.


In [ ]:
verify_gate(exp)
